In [ ]:
%pip install engineeringkit


# North's matched filter (1943)

Working on radar at RCA Laboratories, Dwight North asked which receiving
filter makes a known pulse stand out best above white noise. His wartime
report answered it: the filter whose impulse response is the pulse
reversed in time (and conjugated), :func:`~engineeringkit.communications.matched_filter`.
Sampled at the end of the pulse, its output signal-to-noise ratio is
$2E/N_0$ (:func:`~engineeringkit.communications.matched_filter_snr`).
That depends only on the pulse's energy, not its shape, and no linear
filter does better: the bound is the Cauchy-Schwarz inequality.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from engineeringkit.communications import matched_filter, matched_filter_snr

## Pulling a chirp out of noise



In [ ]:
rng = np.random.default_rng(1943)
fs = 1e6
n = np.arange(200)
pulse = np.cos(2 * np.pi * (5e3 * n / fs + 0.5 * 4e8 * (n / fs) ** 2))
energy = np.sum(pulse**2) / fs  # [J] on a 1-ohm load
N0 = 2e-7  # one-sided noise density [W/Hz]
sigma = np.sqrt(N0 / 2 * fs)  # sample standard deviation for two-sided N0/2
received = sigma * rng.standard_normal(2000)
start = 900
received[start : start + pulse.size] += pulse
output = np.convolve(received, matched_filter(pulse)) / fs
peak = start + pulse.size - 1

predicted = float(matched_filter_snr(energy, N0))
noise_out = np.convolve(sigma * rng.standard_normal((50, 2000)).ravel(), matched_filter(pulse))[: 50 * 2000] / fs
measured = output[peak] ** 2 / np.var(noise_out)
print(f"predicted 2E/N0 = {10 * np.log10(predicted):.2f} dB, measured {10 * np.log10(measured):.2f} dB")
assert abs(10 * np.log10(measured / predicted)) < 0.5
assert np.argmax(np.abs(output)) == peak

## No other filter does better

Compare the output SNR of random filters and of a plain boxcar averager.



In [ ]:
def snr(filt):
    return np.dot(pulse, filt[::-1]) ** 2 / (sigma**2 * np.sum(filt**2))


h = matched_filter(pulse)
others = [snr(rng.standard_normal(pulse.size)) for _ in range(500)] + [snr(np.ones(pulse.size))]
assert np.isclose(snr(h), predicted) and max(others) < predicted
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(received, lw=0.5, label="received")
axes[0].plot(output / np.max(np.abs(output)) * 4 * sigma, label="matched-filter output (scaled)")
axes[0].axvline(peak, color="k", ls=":")
axes[0].set(xlabel="sample", title="The pulse is invisible until filtered")
axes[0].legend()
axes[1].hist(10 * np.log10(others), bins=30, label="random filters")
axes[1].axvline(10 * np.log10(predicted), color="C3", label="matched filter, 2E/N0")
axes[1].set(xlabel="output SNR [dB]", ylabel="count")
axes[1].legend()
fig.tight_layout()

plt.show()